# Day 21: Date & Time Handling in Pandas

## Converting to Datetime
Dates loaded from a CSV or database often come in as plain text (an `object` dtype in pandas), which means pandas has no idea it's looking at a date — you can't compare, sort chronologically, or do date math on strings the way you can with real datetime values. `pd.to_datetime()` parses those text values into pandas' actual datetime type, which is a required first step before any date-based filtering, arithmetic, or extraction (all covered in the cells below). This is a step you'll do at the start of nearly every project that involves time-based data. The output here is `df.dtypes`, and after the conversion the `Date` column should show as `datetime64[ns]` instead of `object`, confirming the conversion worked.</cell id="cell-1">


In [ ]:
import pandas as pd

df = pd.DataFrame({'Date': ['2024-01-05', '2024-02-14', '2024-03-21']})
df['Date'] = pd.to_datetime(df['Date'])  # parse text strings into real datetime values
print(df.dtypes)  # confirms 'Date' is now datetime64[ns], not a plain string/object

## The .dt Accessor
Once a column is a real datetime type, the `.dt` accessor unlocks vectorized access to its individual components — year, month, day, weekday name, hour, and more — applied across the whole column at once, similar to how `.str` works for text. This is essential for time-based feature engineering: for example, a "day of week" or "month" column can reveal patterns (like weekend sales spikes or seasonal trends) that the raw date alone doesn't make obvious to a model. Below we pull out the year, the month number, and the full weekday name into three new columns. In the output, check that Year/Month line up with the original dates and that Weekday correctly names the day (e.g. 2024-01-05 was a Friday).</cell id="cell-3">


In [ ]:
df['Year'] = df['Date'].dt.year  # extract the year as an integer
df['Month'] = df['Date'].dt.month  # extract the month as an integer (1-12)
df['Weekday'] = df['Date'].dt.day_name()  # extract the full weekday name, e.g. 'Friday'
print(df)

## Date Arithmetic
Once dates are proper datetime values, pandas lets you do arithmetic on them directly — adding a fixed span of time to shift a date forward, or subtracting two dates to get the elapsed time between them. `pd.Timedelta` represents a duration (like "7 days") that can be added to or subtracted from a datetime column, and subtracting one datetime column from another produces a `Timedelta` column, from which `.dt.days` extracts just the number of whole days. This is how you'd compute things like "renewal date" (today + 30 days) or "days since signup" (today - signup_date) in a real dataset. In the output, `Next_Week` should be exactly 7 days after each `Date`, and `Days_Since_Start` should be 0 for the earliest date and increase from there.</cell id="cell-5">


In [ ]:
# add a fixed 7-day duration to every date
df['Next_Week'] = df['Date'] + pd.Timedelta(days=7)
# subtract dates to get a Timedelta, then .dt.days pulls out just the day count as an integer
df['Days_Since_Start'] = (df['Date'] - df['Date'].min()).dt.days
print(df)

## Filtering by Date Range
A very common real-world need is restricting analysis to a specific time window — last quarter's sales, this year's signups, a particular campaign period. Because the `Date` column is a real datetime type, you can compare it directly to date strings like `'2024-02-01'` and pandas will parse them for the comparison, producing a boolean mask just like any other conditional filter. Combining two conditions with `&` (both must be true) restricts the mask to a range, mirroring how you'd filter numeric columns with `(df['x'] > 1) & (df['x'] < 10)`. The output shows only the rows whose `Date` falls between Feb 1 and Mar 1, 2024 (inclusive) — here, just the February 14th row.</cell id="cell-7">


In [ ]:
# boolean mask: True where Date falls within Feb 1 - Mar 1, 2024 (pandas auto-parses the date strings)
mask = (df['Date'] >= '2024-02-01') & (df['Date'] <= '2024-03-01')
print(df[mask])